In [ ]:
import os
import pandas as pd
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt
import csv
import cv2
from scipy.spatial.distance import euclidean
from retinaface import RetinaFace
from matplotlib.ticker import FormatStrFormatter

plt.rcParams.update({
    'axes.titlesize': 16,
    'axes.labelsize': 14,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'legend.fontsize': 12
})

In [ ]:
# remove pure black images (NSFW in dreambooth)

image_dir = '/data_aug/synthetic/synthetic_train_realistic'

for class_name in tqdm(os.listdir(image_dir)):
    # Path to the folder with your images
    image_folder = os.path.join(image_dir, class_name)

    for filename in os.listdir(image_folder):
        if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
            img_path = os.path.join(image_folder, filename)
            try:
                img = Image.open(img_path).convert("RGB")
                np_img = np.array(img)

                if np.all(np_img == 0):  # pure black
                    os.remove(img_path)
            except Exception as e:
                print(f"Error processing {img_path}: {e}")


In [ ]:
# resize from 1024 x 1024 to 512 x 512
# from PIL import Image
# import os

# for i in ['eval_10000', 'eval_20000', 'eval_30000', 'eval_40000', 'eval_50000', 'eval_60000', 'eval_70000', 'eval_80000']:
#     input_path = f"/FastGAN-pytorch/output_log_train/train_results/test1/{i}/img"
#     output_path = f"/FastGAN-pytorch/output_log_train/train_results/test1/{i}/img"
#     os.makedirs(output_path, exist_ok=True)

#     for img_name in tqdm(os.listdir(input_path)):
#         if img_name.lower().endswith((".png", ".jpg", ".jpeg")):
#             img = Image.open(os.path.join(input_path, img_name)).convert("RGB")
#             img_resized = img.resize((512, 512), resample=Image.LANCZOS)
#             img_resized.save(os.path.join(output_path, img_name))


In [ ]:
import pandas as pd

# Example: read your CSV
df = pd.read_csv("/data_aug/landmark_distances.csv")

# Extract class name before the first dot in 'image_name'
df['class'] = df['image_name'].str.extract(r'^([^\.]+)\.\d+')

# Group by class and compute the mean of all other (landmark) columns
mean_landmarks = df.groupby('class').mean(numeric_only=True)


## Landmark detection

In [ ]:
# pad image for better retina face recognition
def pad_image(img, pad_ratio=0.5):
    h, w = img.shape[:2]
    pad_h, pad_w = int(h * pad_ratio), int(w * pad_ratio)
    padded_img = cv2.copyMakeBorder(img, pad_h, pad_h, pad_w, pad_w, borderType=cv2.BORDER_CONSTANT, value=[0, 0, 0])
    return padded_img

synthetic_dir = "/data_aug/synthetic/synthetic_train_realistic"
for class_name in tqdm(os.listdir(synthetic_dir)):
    # pad images
    # generate metadata including landmarks for each faces into a csv file
    class_dir = f"/data_aug/synthetic/synthetic_train_realistic/{class_name}"
    padded_dir = f"/data_aug/synthetic/synthetic_padded_realistic/{class_name}"
    os.makedirs(padded_dir, exist_ok=True)
    class_paths = [os.path.join(class_dir, item) for item in os.listdir(class_dir) if os.path.isfile(os.path.join(class_dir, item))]
    for path in class_paths:
        img = cv2.imread(path)
        padded_img = pad_image(img, pad_ratio=0.5)
        cv2.imwrite(os.path.join(padded_dir, os.path.basename(path)), padded_img)

    # create csv files for each class
    csv_file = f"/data_aug/synthetic/synthetic_meta_train_realistic/{class_name}.csv"
    fieldnames = [
        "image_name", "face_id", "score",
        "facial_area_x1", "facial_area_y1", "facial_area_x2", "facial_area_y2",
        "right_eye_x", "right_eye_y",
        "left_eye_x", "left_eye_y",
        "nose_x", "nose_y",
        "mouth_right_x", "mouth_right_y",
        "mouth_left_x", "mouth_left_y"
    ]
    # Open CSV once and append rows
    with open(csv_file, mode='w', newline='') as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()

        class_paths = [os.path.join(padded_dir, item) for item in os.listdir(padded_dir) if os.path.isfile(os.path.join(padded_dir, item))]

        for path in class_paths:
            face_info = RetinaFace.detect_faces(path, threshold=0.9)

            if isinstance(face_info, dict):
                for face_id, data in face_info.items():
                    row = {
                        "image_name": os.path.basename(path),
                        "face_id": face_id,
                        "score": data.get("score", ""),
                        "facial_area_x1": data["facial_area"][0],
                        "facial_area_y1": data["facial_area"][1],
                        "facial_area_x2": data["facial_area"][2],
                        "facial_area_y2": data["facial_area"][3],
                        "right_eye_x": data["landmarks"]["right_eye"][0], # note: right eye is on the left side of the image
                        "right_eye_y": data["landmarks"]["right_eye"][1],
                        "left_eye_x": data["landmarks"]["left_eye"][0], # note: left eye is on the right side of the image
                        "left_eye_y": data["landmarks"]["left_eye"][1],
                        "nose_x": data["landmarks"]["nose"][0],
                        "nose_y": data["landmarks"]["nose"][1],
                        "mouth_right_x": data["landmarks"]["mouth_right"][0],
                        "mouth_right_y": data["landmarks"]["mouth_right"][1],
                        "mouth_left_x": data["landmarks"]["mouth_left"][0],
                        "mouth_left_y": data["landmarks"]["mouth_left"][1],
                    }
                    writer.writerow(row)
            else:
                print(f"Face detection failed for {path}")
                continue

In [ ]:
# # remove face_id != 1 (multiple faces in one image) rows
# for csvfile in os.listdir('/data_aug/synthetic/synthetic_meta_train_realistic'):
#     if csvfile.endswith('.csv') and csvfile.startswith('eval_'):
#         df = pd.read_csv(f"/data_aug/synthetic/synthetic_meta_train_realistic/{csvfile}")
#         # remove face_id != 1
#         df = df[df['face_id'] == 'face_1']
#         df.to_csv(f"/data_aug/synthetic/synthetic_meta_train_realistic/{csvfile}", index=False)

In [ ]:
# count the number of images in total in each subfolder
image_dir = '/data_aug/synthetic/synthetic_train_realistic'
image_counts = 0
for class_name in os.listdir(image_dir):
    # Path to the folder with your images
    image_folder = os.path.join(image_dir, class_name)
    for filename in os.listdir(image_folder):
        if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
            image_counts += 1

print(f"Total number of images: {image_counts}")

# faces can be detected in the images score > 99%
meta_dir = '/data_aug/synthetic/synthetic_meta_train_realistic'
face_count = 0
row_count = 0
for csv_files in os.listdir(meta_dir):
    if csv_files.endswith('.csv'):
        csv_path = os.path.join(meta_dir, csv_files)
        df = pd.read_csv(csv_path)
        threshold = 0.99
        valid_faces = df[df['score'] > threshold]
        face_count += len(valid_faces)
        row_count += len(df)

print(f"Total number of faces detected with score > {threshold}: {face_count}")
print(f"Total number of rows in all CSV files: {row_count}")


In [ ]:
# pad image for better retina face recognition
def pad_image(img, pad_ratio=0.5):
    h, w = img.shape[:2]
    pad_h, pad_w = int(h * pad_ratio), int(w * pad_ratio)
    padded_img = cv2.copyMakeBorder(img, pad_h, pad_h, pad_w, pad_w, borderType=cv2.BORDER_CONSTANT, value=[0, 0, 0])
    return padded_img

synthetic_dir = "/FastGAN-pytorch/output_log_train/train_results/test1"
loops = ['eval_10000', 'eval_20000', 'eval_30000', 'eval_40000', 'eval_50000', 'eval_60000', 'eval_70000', 'eval_80000']
for class_name in loops:
    # pad images
    # generate metadata including landmarks for each faces into a csv file
    class_dir = f"/FastGAN-pytorch/output_log_train/train_results/test1/{class_name}/img"
    padded_dir = f"/data_aug/synthetic/fastgan_padded/{class_name}"
    os.makedirs(padded_dir, exist_ok=True)
    class_paths = [os.path.join(class_dir, item) for item in os.listdir(class_dir)]
    for path in class_paths:
        img = cv2.imread(path)
        padded_img = pad_image(img, pad_ratio=0.5)
        cv2.imwrite(os.path.join(padded_dir, os.path.basename(path)), padded_img)

    # create csv files for each class
    csv_file = f"/data_aug/synthetic/fastgan_meta/{class_name}.csv"
    fieldnames = [
        "image_name", "face_id", "score",
        "facial_area_x1", "facial_area_y1", "facial_area_x2", "facial_area_y2",
        "right_eye_x", "right_eye_y",
        "left_eye_x", "left_eye_y",
        "nose_x", "nose_y",
        "mouth_right_x", "mouth_right_y",
        "mouth_left_x", "mouth_left_y"
    ]
    # Open CSV once and append rows
    with open(csv_file, mode='w', newline='') as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()

        class_paths = [os.path.join(padded_dir, item) for item in os.listdir(padded_dir) if os.path.isfile(os.path.join(padded_dir, item))]

        for path in tqdm(class_paths):
            face_info = RetinaFace.detect_faces(path, threshold=0.9)

            if isinstance(face_info, dict):
                for face_id, data in face_info.items():
                    row = {
                        "image_name": os.path.basename(path),
                        "face_id": face_id,
                        "score": data.get("score", ""),
                        "facial_area_x1": data["facial_area"][0],
                        "facial_area_y1": data["facial_area"][1],
                        "facial_area_x2": data["facial_area"][2],
                        "facial_area_y2": data["facial_area"][3],
                        "right_eye_x": data["landmarks"]["right_eye"][0], # note: right eye is on the left side of the image
                        "right_eye_y": data["landmarks"]["right_eye"][1],
                        "left_eye_x": data["landmarks"]["left_eye"][0], # note: left eye is on the right side of the image
                        "left_eye_y": data["landmarks"]["left_eye"][1],
                        "nose_x": data["landmarks"]["nose"][0],
                        "nose_y": data["landmarks"]["nose"][1],
                        "mouth_right_x": data["landmarks"]["mouth_right"][0],
                        "mouth_right_y": data["landmarks"]["mouth_right"][1],
                        "mouth_left_x": data["landmarks"]["mouth_left"][0],
                        "mouth_left_y": data["landmarks"]["mouth_left"][1],
                    }
                    writer.writerow(row)
            else:
                print(f"Face detection failed for {path}")
                continue

In [ ]:
# # remove face_id != 1 (multiple faces in one image) rows
# for csvfile in os.listdir('/data_aug/synthetic/fastgan_meta'):
#     if csvfile.endswith('.csv') and csvfile.startswith('eval_'):
#         df = pd.read_csv(f"/data_aug/synthetic/fastgan_meta/{csvfile}")
#         # remove face_id != 1
#         df = df[df['face_id'] == 'face_1']
#         df.to_csv(f"/data_aug/synthetic/fastgan_meta/{csvfile}", index=False)

In [ ]:
# check retinaface's performance
fastgan_dir = '/FastGAN-pytorch/output_log_train/train_results/test1'
loops = ['eval_10000', 'eval_20000', 'eval_30000', 'eval_40000', 'eval_50000', 'eval_60000', 'eval_70000', 'eval_80000']
img_count = 0
for eval_loop in loops:
    image_dir = f"{fastgan_dir}/{eval_loop}/img"
    for img in os.listdir(image_dir):
        if img.lower().endswith(('.png', '.jpg', '.jpeg')):
            img_count += 1
print(f"Total number of images: {img_count}")

fastgan_meta_dir = "/data_aug/synthetic/fastgan_meta"
face_count = 0
row_count = 0
for csv_file in os.listdir(fastgan_meta_dir):
    if csv_file.endswith('.csv') and csv_file.startswith('eval_'):
        csv_path = os.path.join(fastgan_meta_dir, csv_file)
        df = pd.read_csv(csv_path)
        threshold = 0.99
        valid_faces = df[df['score'] > threshold]
        face_count += len(valid_faces)
        row_count += len(df)
print(f"Total number of faces detected with score > {threshold}: {face_count}")
print(f"Total number of rows in all CSV files: {row_count}")


## Cosine Similarity comparsion between real and synthetic images

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from scipy.spatial.distance import cdist, euclidean
from tqdm import tqdm

# Load real landmark statistics
df_real = pd.read_csv("/data_aug/landmark_distances.csv")
df_real['class'] = df_real['image_name'].str.extract(r'^([^\.]+)\.\d+')
mean_landmarks = df_real.groupby('class').mean(numeric_only=True)
class_list = df_real['class'].unique()

# Synthetic data directories
synthetic_dir = "/data_aug/synthetic/synthetic_train_realistic"
synthetic_meta_dir = "/data_aug/synthetic/synthetic_meta_train_realistic"

# Landmark definitions
landmark_names = [
    ('right_eye_x', 'right_eye_y'),
    ('left_eye_x', 'left_eye_y'),
    ('nose_x', 'nose_y'),
    ('mouth_right_x', 'mouth_right_y'),
    ('mouth_left_x', 'mouth_left_y')
]

# Helper: normalized landmark distance matrix
def compute_normalized_landmark_matrix(row):
    x1, y1 = row['facial_area_x1'], row['facial_area_y1']
    x2, y2 = row['facial_area_x2'], row['facial_area_y2']
    w, h = x2 - x1, y2 - y1

    if w == 0 or h == 0:
        return np.full((5, 5), np.nan)

    landmarks = np.array([
        [(row[x] - x1) / w, (row[y] - y1) / h]
        for x, y in landmark_names
    ])

    dist_matrix = np.zeros((5, 5))
    for i in range(5):
        for j in range(5):
            dist_matrix[i, j] = euclidean(landmarks[i], landmarks[j])

    return dist_matrix

# Store per-class average rank metrics
class_landmark_similarity = {}

# Iterate through each class's synthetic data
for class_real_name in tqdm(os.listdir(synthetic_dir)):
    meta_path = os.path.join(synthetic_meta_dir, class_real_name)
    if not meta_path.endswith(".csv"):
        continue

    df = pd.read_csv(meta_path)
    distance_matrices = df.apply(compute_normalized_landmark_matrix, axis=1)

    if distance_matrices.empty:
        df.insert(1, 'cos', None)
        df.insert(2, 'euc', None)
        df.insert(3, 'cosine_similarity_rank', None)
        df.insert(4, 'euclidean_distance_rank', None)
        df.to_csv(meta_path, index=False)
        continue

    cos_vals = []
    euc_vals = []
    cos_ranks = []
    euc_ranks = []

    for sample in distance_matrices:
        sample_flat = sample.reshape(1, -1)

        # True class similarity
        true_mean = mean_landmarks.loc[class_real_name].values.reshape(1, -1)
        cos_vals.append(cosine_similarity(true_mean, sample_flat)[0, 0])
        euc_vals.append(cdist(sample_flat, true_mean, metric='euclidean')[0, 0])

        # Rank across all classes
        cos_sims = []
        euc_dists = []

        for class_name in class_list:
            class_mean = mean_landmarks.loc[class_name].values.reshape(1, -1)
            cos_sims.append(cosine_similarity(class_mean, sample_flat)[0, 0])
            euc_dists.append(cdist(sample_flat, class_mean, metric='euclidean')[0, 0])

        cos_series = pd.Series(cos_sims, index=class_list).sort_values(ascending=False)
        euc_series = pd.Series(euc_dists, index=class_list).sort_values(ascending=True)

        cos_ranks.append(cos_series.index.get_loc(class_real_name))
        euc_ranks.append(euc_series.index.get_loc(class_real_name))

    # Insert metrics
    df.insert(1, 'cos', cos_vals)
    df.insert(2, 'euc', euc_vals)
    df.insert(3, 'cosine_similarity_rank', cos_ranks)
    df.insert(4, 'euclidean_distance_rank', euc_ranks)
    df.to_csv(meta_path, index=False)
    
    # class_landmark_similarity[class_real_name] = [
    #     np.mean(cos_ranks),
    #     np.mean(euc_ranks)
    # ]


In [ ]:
from lpips import LPIPS
from PIL import Image
import torch
from torchvision import transforms

similarity = LPIPS(net='alex').to(device='cuda')

meta_dir = "/data_aug/synthetic/synthetic_meta_train_realistic"
synthetic_dir = '/data_aug/synthetic/synthetic_train_realistic'

transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

for class_name in tqdm(os.listdir(synthetic_dir)):
    # assign label based on the simliarity towards closest class
    df = pd.read_csv(f"{meta_dir}/{class_name}.csv")
    image_names = df['image_name']
    lpips_score = []
    image_dir = os.path.join(synthetic_dir, class_name)
    for img_name in image_names:
        img = transform(Image.open(os.path.join(image_dir, img_name))).unsqueeze(0).to(device='cuda')
        label_path = f'/data_aug/train/{class_name}'
        score = []
        for label_images in os.listdir(label_path):
            label_img = transform(Image.open(os.path.join(label_path, label_images))).unsqueeze(0).to(device='cuda')
            score.append(similarity(img, label_img).item())
        
        if np.mean(score) > 0:
            lpips_score.append(np.mean(score))

    if len(lpips_score) > 0:
        df.insert(1, 'lpips_score', lpips_score)
        df.to_csv(os.path.join(meta_dir, f"{class_name}.csv"), index=False)
    else:
        df.insert(1, 'lpips_score', None)
        df.to_csv(os.path.join(meta_dir, f"{class_name}.csv"), index=False)



In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
from scipy.spatial.distance import cdist, euclidean
import pandas as pd
import numpy as np
import os

# Load the real landmark distances to compute class means
df_real = pd.read_csv("/data_aug/landmark_distances.csv")
df_real['class'] = df_real['image_name'].str.extract(r'^([^\.]+)\.\d+')
mean_landmarks = df_real.groupby('class').mean(numeric_only=True)
class_list = df_real['class'].unique()

# Meta directory for synthetic data
fastgan_meta_dir = "/data_aug/synthetic/fastgan_meta"
loops = ['eval_10000', 'eval_20000', 'eval_30000', 'eval_40000',
         'eval_50000', 'eval_60000', 'eval_70000', 'eval_80000']

# Landmark column names
landmark_names = [
    ('right_eye_x', 'right_eye_y'),
    ('left_eye_x', 'left_eye_y'),
    ('nose_x', 'nose_y'),
    ('mouth_right_x', 'mouth_right_y'),
    ('mouth_left_x', 'mouth_left_y')
]

# Helper function to compute normalized landmark distance matrix
def compute_normalized_landmark_matrix(row):
    x1, y1 = row['facial_area_x1'], row['facial_area_y1']
    x2, y2 = row['facial_area_x2'], row['facial_area_y2']
    w, h = x2 - x1, y2 - y1

    if w == 0 or h == 0:
        return np.full((5, 5), np.nan)

    landmarks = np.array([
        [(row[x] - x1) / w, (row[y] - y1) / h]
        for x, y in landmark_names
    ])

    dist_matrix = np.zeros((5, 5))
    for i in range(5):
        for j in range(5):
            dist_matrix[i, j] = euclidean(landmarks[i], landmarks[j])

    return dist_matrix

for loop in loops:
    df = pd.read_csv(os.path.join(fastgan_meta_dir, f"{loop}.csv"))
    distance_matrices = df.apply(compute_normalized_landmark_matrix, axis=1)

    # Label assignment based on closest class
    disease_label_cos, disease_label_euc = [], []
    for sample in distance_matrices:
        sample_flat = sample.reshape(1, -1)

        cos_sims = [cosine_similarity(mean_landmarks.loc[c].values.reshape(1, -1), sample_flat)[0, 0]
                    for c in class_list]
        euc_dists = [cdist(sample_flat, mean_landmarks.loc[c].values.reshape(1, -1), metric='euclidean')[0, 0]
                     for c in class_list]

        disease_label_cos.append(class_list[np.argmax(cos_sims)])
        disease_label_euc.append(class_list[np.argmin(euc_dists)])

    # Assign the new labels
    df.insert(1, 'disease_label_cos', disease_label_cos)
    df.insert(2, 'disease_label_euc', disease_label_euc)

    # Compute similarity/distance to assigned class
    cos_sim_vals, euc_dist_vals = [], []
    for sample, cos_label, euc_label in zip(distance_matrices, disease_label_cos, disease_label_euc):
        sample_flat = sample.reshape(1, -1)
        mean_cos = mean_landmarks.loc[cos_label].values.reshape(1, -1)
        mean_euc = mean_landmarks.loc[euc_label].values.reshape(1, -1)

        cos_sim_vals.append(cosine_similarity(mean_cos, sample_flat)[0, 0])
        euc_dist_vals.append(cdist(sample_flat, mean_euc, metric='euclidean')[0, 0])

    df.insert(3, 'cos', cos_sim_vals)
    df.insert(4, 'euc', euc_dist_vals)

    df.to_csv(os.path.join(fastgan_meta_dir, f"{loop}.csv"), index=False)


## LPIPS score calculation

In [ ]:
from lpips import LPIPS
from PIL import Image
import torch
from torchvision import transforms
import os
import numpy as np
import pandas as pd

similarity = LPIPS(net='alex').to(device='cuda')

fastgan_meta_dir = "/data_aug/synthetic/fastgan_meta"
fastgan_dir = '/FastGAN-pytorch/output_log_train/train_results/test1'
loops = ['eval_10000', 'eval_20000', 'eval_30000', 'eval_40000', 'eval_50000', 'eval_60000', 'eval_70000', 'eval_80000']

transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

for loop in tqdm(loops):
    # assign label based on the simliarity towards closest class
    df = pd.read_csv(f"{fastgan_meta_dir}/{loop}.csv")
    image_names = df['image_name']
    image_label_cos = df['disease_label_cos']
    image_label_euc = df['disease_label_euc']
    image_dir = os.path.join(fastgan_dir, loop, 'img')
    lpips_score_cos = []
    lpips_score_euc = []
    for img_name, label_cos, label_euc in zip(image_names, image_label_cos, image_label_euc):
        img = transform(Image.open(os.path.join(image_dir, img_name))).unsqueeze(0).to(device='cuda')
        label_cos_path = f"/data_aug/train/{label_cos}"
        label_euc_path = f"/data_aug/train/{label_euc}"
        score_cos = []
        score_euc = []
        for label_images in os.listdir(label_cos_path):
            label_img = transform(Image.open(os.path.join(label_cos_path, label_images))).unsqueeze(0).to(device='cuda')
            score_cos.append(similarity(img, label_img).item())
        for label_images in os.listdir(label_euc_path):
            label_img = transform(Image.open(os.path.join(label_euc_path, label_images))).unsqueeze(0).to(device='cuda')
            score_euc.append(similarity(img, label_img).item())
        lpips_score_cos.append(np.mean(score_cos))
        lpips_score_euc.append(np.mean(score_euc))
    cols_to_drop = ['lpips_score_cos', 'lpips_score_euc']
    # df.drop(columns=cols_to_drop, inplace=True)
    df.insert(3, 'lpips_score_cos', lpips_score_cos)
    df.insert(4, 'lpips_score_euc', lpips_score_euc)
    df.to_csv(os.path.join(fastgan_meta_dir, f"{loop}.csv"), index=False)



In [ ]:
# read csv
df = pd.read_csv("/data_aug/landmark_distances.csv")

# Extract class name before the first dot in 'image_name'
df['class'] = df['image_name'].str.extract(r'^([^\.]+)\.\d+')

# Group by class and compute the mean of all other (landmark) columns
mean_landmarks = df.groupby('class').mean(numeric_only=True)

class_list = df['class'].unique()

# synthetic meta and images dir
synthetic_dir = "/data_aug/synthetic/synthetic_train_realistic"
synthetic_meta_dir = "/data_aug/synthetic/synthetic_meta_train_realistic"

image_names_all = []
cos_all = []
euc_all = []
score_all = []
cos_rank_all = []
euc_rank_all = []
lpips_all = []

for csvfile in os.listdir(synthetic_meta_dir):
    df = pd.read_csv(os.path.join(synthetic_meta_dir, csvfile))
    image_names = df['image_name'].tolist()
    cos = df['cos'].tolist()
    euc = df['euc'].tolist()
    score = df['score'].tolist()
    cos_rank = df['cosine_similarity'].tolist()
    euc_rank = df['euclidean_distance'].tolist()
    lpips = df['lpips_score'].tolist()

    image_names_all.extend(image_names)
    cos_all.extend(cos)
    euc_all.extend(euc)
    score_all.extend(score)
    cos_rank_all.extend(cos_rank)
    euc_rank_all.extend(euc_rank)
    lpips_all.extend(lpips)
 
# make sure the length of all lists are the same
print(len(image_names_all))
print(len(cos_all))
print(len(euc_all))
print(len(score_all))
print(len(cos_rank_all))
print(len(euc_rank_all))
print(len(lpips_all))

# order top n of the cos_all based on the similarity towards the class mean
topk_scores = []
topk_distance_rank = []
topk_lpips = []
topk_image_names = []

for x in [1000, 2000, 3000, 4000, 5000, 6000]:
    top_indices = np.argsort(cos_all)[-x:][::-1]  # Get the indices of the top n scores
    # Get the meta data
    top_image_names = [image_names_all[i] for i in top_indices]
    top_image_scores = [score_all[i] for i in top_indices]
    top_distance = [cos_all[i] for i in top_indices]
    top_distance_rank = [cos_rank_all[i] for i in top_indices]
    top_lpips = [lpips_all[i] for i in top_indices]

    # write above to a csv
    # Create a DataFrame
    df_top = pd.DataFrame({
        'image_name': top_image_names,
        'score': top_image_scores,
        'distance': top_distance,
        'rank': top_distance_rank,
        'lpips': top_lpips
    })

    # Save to CSV
    df_top.to_csv(f'/data_aug/synthetic/synthetic_train_realistic_topk_meta/top_{x}_cos_metadata.csv', index=False)

import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FormatStrFormatter
import matplotlib.ticker as mticker


topk_scores = []
topk_lpips = []
cutoffs =  [1000, 2000, 4000, 6000]

for x in cutoffs:
    df = pd.read_csv(f'/data_aug/synthetic/synthetic_train_realistic_topk_meta/top_{x}_cos_metadata.csv')
    topk_scores.append(df['score'].mean())
    topk_lpips.append(df['lpips'].mean())

plt.figure(figsize=(12, 4))

# Subplot 1: Confidence Score
plt.subplot(1, 2, 1)
plt.plot(cutoffs, topk_scores, marker='o')
plt.title('Top-n cutoff vs. Confidence score')
plt.xlabel('n')
plt.xticks(cutoffs)
plt.grid()

# Subplot 2: LPIPS with 4 decimal formatting
ax2 = plt.subplot(1, 2, 2)
ax2.plot(cutoffs, topk_lpips, marker='o')
ax2.set_title('Top-n cutoff vs. LPIPS')
ax2.set_xlabel('n')
ax2.set_ylabel('LPIPS')
ax2.set_xticks(cutoffs)
ax2.yaxis.set_major_formatter(FormatStrFormatter('%.4f'))
ax2.grid()

plt.tight_layout()
plt.savefig('/figures/dreambooth_topn.png', dpi=300, bbox_inches='tight', transparent=True)
plt.show()


# Top-$n$ data based on cosine similarity

In [ ]:
# calculate the distance between the class mean and synthetic data
from sklearn.metrics.pairwise import cosine_similarity
from scipy.spatial.distance import cdist

# Example: read your CSV
df = pd.read_csv("/data_aug/landmark_distances.csv")

# Extract class name before the first dot in 'image_name'
df['class'] = df['image_name'].str.extract(r'^([^\.]+)\.\d+')

# Group by class and compute the mean of all other (landmark) columns
mean_landmarks = df.groupby('class').mean(numeric_only=True)

class_list = df['class'].unique()

fastgan_meta_dir = "/data_aug/synthetic/fastgan_meta"

cutoffs = [1000, 2000, 3000, 4000, 5000, 6000]

image_names_all = []
cos_all = []
euc_all = []
score_all = []
lpips_cos_all = []
lpips_euc_all = []
labels_cos_all = []
labels_euc_all = []

for csv_file in os.listdir(fastgan_meta_dir):
    # assign label based on the simliarity towards closest class
    if not csv_file.endswith('.csv') or not csv_file.startswith('eval_'):
        continue
    df = pd.read_csv(f"{fastgan_meta_dir}/{csv_file}")

    image_names = df['image_name'].tolist()
    csv_file_name = csv_file.split('.')[0]
    image_names = [f"{csv_file_name}/img/{name}" for name in image_names]
    cos = df['cos'].tolist()
    euc = df['euc'].tolist()
    score = df['score'].tolist()
    lpips_cos = df['lpips_score_cos'].tolist()
    lpips_euc = df['lpips_score_euc'].tolist()
    labels_cos = df['disease_label_cos'].tolist()
    labels_euc = df['disease_label_euc'].tolist()

    image_names_all.extend(image_names)
    cos_all.extend(cos)
    euc_all.extend(euc)
    score_all.extend(score)
    lpips_cos_all.extend(lpips_cos)
    lpips_euc_all.extend(lpips_euc)
    labels_cos_all.extend(labels_cos)
    labels_euc_all.extend(labels_euc)

# order top n of the cos_all based on the similarity towards the class mean

for x in [1000, 2000, 3000, 4000, 5000, 6000]:
    top_indices = np.argsort(cos_all)[-x:][::-1]  # Get the indices of the top n scores
    # Get the meta data
    top_image_names = [image_names_all[i] for i in top_indices]
    top_image_scores = [score_all[i] for i in top_indices]
    top_distance = [cos_all[i] for i in top_indices]
    top_lpips = [lpips_cos_all[i] for i in top_indices]
    top_labels = [labels_cos_all[i] for i in top_indices]

    # write above to a csv
    # Create a DataFrame
    df_top = pd.DataFrame({
        'image_name': top_image_names,
        'score': top_image_scores,
        'distance': top_distance,
        'lpips': top_lpips,
        'label': top_labels
    })

    # Save to CSV
    df_top.to_csv(f'/data_aug/synthetic/fastgan_meta/top_{x}_cos_metadata.csv', index=False)


import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FormatStrFormatter

topk_scores = []
topk_lpips = []
cutoffs = [1000, 2000, 4000, 6000]
for x in cutoffs:
    # read csv files 
    df = pd.read_csv(f'/data_aug/synthetic/fastgan_meta/top_{x}_cos_metadata.csv')
    topk_scores.append(df['score'].mean())
    topk_lpips.append(df['lpips'].mean())

plt.figure(figsize=(12, 4))

# Subplot 1: Confidence Score
plt.subplot(1, 2, 1)
plt.plot(cutoffs, topk_scores, marker='o')
plt.title('Top-n cutoff vs. Confidence score')
plt.xlabel('n')
plt.ylabel('Confidence score')
plt.xticks(cutoffs)
plt.grid()

# Subplot 2: LPIPS with 4 decimal formatting
ax2 = plt.subplot(1, 2, 2)
ax2.plot(cutoffs, topk_lpips, marker='o')
ax2.set_title('Top-n cutoff vs. LPIPS')
ax2.set_xlabel('n')
ax2.set_ylabel('LPIPS')
ax2.set_xticks(cutoffs)
ax2.yaxis.set_major_formatter(FormatStrFormatter('%.4f'))
ax2.grid()

plt.tight_layout()
plt.savefig('/figures/fastgan_topn.png', dpi=300, bbox_inches='tight', transparent=True)
plt.show()


In [ ]:
# calculate the average LPIPS score for fastgan

# Load the CSV file
df1 = pd.read_csv('/data_aug/synthetic/fastgan_meta/eval_80000.csv')
df2 = pd.read_csv('/data_aug/synthetic/fastgan_meta/eval_70000.csv')
df3 = pd.read_csv('/data_aug/synthetic/fastgan_meta/eval_60000.csv')
df4 = pd.read_csv('/data_aug/synthetic/fastgan_meta/eval_50000.csv')
df5 = pd.read_csv('/data_aug/synthetic/fastgan_meta/eval_40000.csv')
df6 = pd.read_csv('/data_aug/synthetic/fastgan_meta/eval_30000.csv')
df7 = pd.read_csv('/data_aug/synthetic/fastgan_meta/eval_20000.csv')
df8 = pd.read_csv('/data_aug/synthetic/fastgan_meta/eval_10000.csv')
lpips_scores1 = df1['lpips_score_cos']
lpips_scores2 = df2['lpips_score_cos']
lpips_scores3 = df3['lpips_score_cos']
lpips_scores4 = df4['lpips_score_cos']
lpips_scores5 = df5['lpips_score_cos']
lpips_scores6 = df6['lpips_score_cos']
lpips_scores7 = df7['lpips_score_cos']
lpips_scores8 = df8['lpips_score_cos']

lpips_total = []
lpips_total.extend(list(lpips_scores1))
lpips_total.extend(list(lpips_scores2))
lpips_total.extend(list(lpips_scores3))
lpips_total.extend(list(lpips_scores4))
lpips_total.extend(list(lpips_scores5))
lpips_total.extend(list(lpips_scores6))
lpips_total.extend(list(lpips_scores7))
lpips_total.extend(list(lpips_scores8))

print(lpips_total)

print(np.mean(lpips_total))


In [ ]:
# average lpips score for dreambooth
class_lpips = {}
for csv_files in os.listdir("/data_aug/synthetic/synthetic_meta_train_realistic"):
    # split file name
    class_name = csv_files.split('_')[0]
    df = pd.read_csv(os.path.join("/data_aug/synthetic/synthetic_meta_train_realistic", csv_files))
    scores = df['lpips_score']
    if np.mean(scores) > 0:
        class_lpips[class_name] = np.mean(scores)
    else:
        print(class_name)
print(len(class_lpips))

print(np.mean(list(class_lpips.values())))
        

In [ ]:
class_landmark_similarity = {}
for csvfile in os.listdir('/data_aug/synthetic/synthetic_meta_train_realistic'):
    class_name = csvfile.split('.')[0]
    df = pd.read_csv(f"/data_aug/synthetic/synthetic_meta_train_realistic/{csvfile}")
    # remove face_id != 1
    cos_rank = np.mean(df['cosine_similarity'])
    euc_rank = np.mean(df['euclidean_distance'])
    class_landmark_similarity[class_name] = [cos_rank, euc_rank]

plt.figure(figsize=(10, 5))
mean_dis = list(class_landmark_similarity.values())
cos = [v[0] for v in mean_dis]
euc = [v[1] for v in mean_dis]
mean_cos = np.mean(cos)
print(mean_cos)
print(np.mean(euc))
plt.bar(range(len(class_landmark_similarity.keys())), cos, label='Cosine Similarity', alpha=0.6)
plt.xlabel('Disease class')
plt.ylabel('Average true rank')
# plt.title('Average Rank of True Class for DreamBooth-Generated Images')
plt.axhline(y=mean_cos, color='r', linestyle='--', label='Mean Cosine Similarity')
# show the value for axhline on plot mean_cos
plt.text(90, mean_cos+2, f'Mean: {mean_cos:.2f}', color='red', fontsize=12)
plt.savefig('/figures/average_rank.png', dpi=300, bbox_inches='tight', transparent=True)
plt.show()
